# 02 — Telecom materialisation


Translate the native synthetic telecom fixture into physically separate
`SPEC-CORE` and `SPEC-EVAL` outputs. It also records memory and representation
reports. It never overwrites an existing run.


    This is a **thin orchestration notebook**. The tested implementation lives
    in the GitHub package; this notebook only sets paths, calls one workflow,
    and displays its evidence. Run cells from top to bottom.

In [ ]:
# Shared implementation: GitHub in Colab, local source when testing this repository.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/LiliDopidze/anomaly_detection.git"
RUNTIME_REF = os.getenv("ANOMALY_RUNTIME_REF", "main")
LOCAL_SOURCE = os.getenv("ANOMALY_SOURCE_ROOT")

if LOCAL_SOURCE:
    sys.path.insert(0, str(Path(LOCAL_SOURCE).resolve()))
    RUNTIME_COMMIT = "local-working-tree"
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        f"git+{REPOSITORY}@{RUNTIME_REF}",
    ])
    RUNTIME_COMMIT = subprocess.check_output(
        ["git", "ls-remote", REPOSITORY, RUNTIME_REF], text=True
    ).split()[0]

os.environ["ANOMALY_RUNTIME_REF"] = RUNTIME_REF
os.environ["ANOMALY_RUNTIME_COMMIT"] = RUNTIME_COMMIT
print(f"Runtime: {RUNTIME_REF} ({RUNTIME_COMMIT[:12]})")

In [ ]:
# Mount Google Drive in Colab. Local validation skips this block.
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
print(f"Drive root: {DRIVE_ROOT}")

## Configuration

For a first check, set a short `SAMPLE_START` / `SAMPLE_END` or list a
few `ENTITY_IDS`. For the signed-off run, leave them empty to process
the complete fixture. Change `RUN_ID` whenever you want a new run.

In [ ]:
TELECOM_SOURCE = Path(os.getenv(
    "ANOMALY_TELECOM_SOURCE",
    str(DRIVE_ROOT),
))
OUTPUT_ROOT = Path(os.getenv(
    "ANOMALY_OUTPUT_ROOT",
    str(DRIVE_ROOT / "outputs" / "milestone_1" / "v0.3" / "telecom"),
))
RUN_ID = os.getenv("ANOMALY_RUN_ID", "telecom_full_v1")
RUN_ROOT = OUTPUT_ROOT / RUN_ID

SAMPLE_START = os.getenv("ANOMALY_SAMPLE_START") or None
SAMPLE_END = os.getenv("ANOMALY_SAMPLE_END") or None
ENTITY_IDS = tuple(filter(None, os.getenv(
    "ANOMALY_ENTITY_IDS", ""
).split(",")))
BATCH_NATIVE_ROWS = int(os.getenv(
    "ANOMALY_BATCH_NATIVE_ROWS", "250000"
))
MEMORY_BUDGET_GIB = float(os.getenv(
    "ANOMALY_MEMORY_BUDGET_GIB", "8"
))

print(f"Source: {TELECOM_SOURCE}")
print(f"New immutable run: {RUN_ROOT}")

In [ ]:
from anomaly_detection.workflows import contract_summary

summary = contract_summary()
print("SPEC-CORE:", summary["spec_core_tables"])
print("SPEC-EVAL:", summary["spec_eval_tables"])
print("Telecom metrics:", summary["packs"]["telecom"]["metric_count"])

In [ ]:
from anomaly_detection.workflows import materialise_telecom

report = materialise_telecom(
    TELECOM_SOURCE,
    RUN_ROOT,
    sample_start=SAMPLE_START,
    sample_end=SAMPLE_END,
    entity_ids=ENTITY_IDS,
    batch_native_rows=BATCH_NATIVE_ROWS,
    memory_budget_gib=MEMORY_BUDGET_GIB,
)

In [ ]:
from pprint import pprint

pprint({
    "row_counts": report["materialisation"]["row_counts"],
    "peak_rss_gib": report["memory"]["peak_rss_gib"],
    "memory_budget_pass": report["memory"]["budget_pass"],
    "canonical_rows": report["representation"]["canonical_rows"],
    "run_root": report["run_root"],
})
print("Next: run Notebook 03 with the same RUN_ID.")